# Generation at n=200 — corrected

`solution_5_n200.ipynb` with paths fixed so it runs from the repo, from Colab, or from anywhere
with network access — and with the defects found while going through it repaired.

## What was wrong

**1. Paths.** The original opens `corpus_v2.json` and `qa_pairs_wiki.json` from the working
directory. In this repo those are `data/corpus.json` and `data/qa_pairs_wiki.json`. It also wrote
its CSVs to the working directory rather than `results/`, and ended with a bare
`from google.colab import files`, which raises outside Colab. Now: paths are resolved by searching
the repo, then the working directory, then GitHub, and the notebook says which it used.

**2. `parse_judge` drops the correctness verdict when both appear on one line.** The loop used
`if "مدعوم" … elif "مطابق" …`, so a judge reply of `مدعوم: نعم مطابق: نعم` parses as
*(faithful=1, correct=0)* — silently scoring a correct answer as wrong. Verified: 2 of 6 realistic
judge replies parse incorrectly. Now parsed per-field rather than per-line.

**3. Unparseable judge output was indistinguishable from a negative verdict.** Anything the parser
did not recognise returned `(0, 0)` — counted as both unfaithful and incorrect, with nothing in
the results to show it happened. Now tracked as `judge_parsed`, and the parse-failure rate is
reported. If it is not ~0, the metrics below are understated.

**4. Faithfulness conflates refusal with unfaithfulness.** "المعلومة غير متوفرة في النصوص" is not
"supported by the reference texts", so the judge scores refusals essentially at random. In the
committed run this is measurable: of 35 refusals, 19 were scored faithful and 16 unfaithful —
a coin flip on identical behaviour. Faithfulness is now reported **both** including and excluding
refusals, so the headline number is not quietly driven by refusal rate.

**5. The retrieval-failure narrative contradicted its own output.** The text asserting "low refusal
+ low correctness … the generator does not notice the context is wrong and answers confidently
anyway" printed unconditionally — while the run it was printed under showed a refusal rate of
**0.737**, i.e. the opposite, safe-degradation behaviour. The conclusion is now derived from the
measured value.

**6. Smaller things.** `torch_dtype=` is deprecated (it warned in the original output) — replaced
with a post-load cast that works on every version. `torch.cuda.empty_cache()` is guarded.
`truncation_side` is set to `"left"` and truncation is counted and reported.

## On truncation — checked, and *not* currently a bug

Worth stating precisely, since it looks alarming: with `max_length=3072` and right-side
truncation, an overflowing judge prompt would lose its trailing output-format instruction and the
judge would emit free text, which defect 3 then scores as (0,0). Measured against this corpus the
worst case is **2845 tokens for the judge prompt and 2694 for the generation prompt** — both
under 3072, so **no prompt was truncated in the original run and its numbers are not affected.**
The margin is only ~7%, though, so the notebook now truncates from the left (dropping context
rather than the instruction) and reports any truncation instead of leaving it silent.

### Install

In [ ]:
import importlib.util, subprocess, sys

need = [p for p, m in [("rank_bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers"),
                       ("transformers", "transformers"), ("accelerate", "accelerate")]
        if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)

import torch
# bitsandbytes only matters on CUDA; skip the install elsewhere so the notebook
# stays runnable on a CPU or non-NVIDIA machine.
if torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=False)
print("deps ready")

### Paths

The original hard-coded `corpus_v2.json` / `qa_pairs_wiki.json` in the working directory. This
resolver walks up for the repo's `data/` directory, then tries the working directory (what a Colab
upload gives you), then falls back to fetching from GitHub — and prints which source it used.
`corpus_v2.json` is committed in this repo as `data/corpus.json`.

In [ ]:
import os, json, urllib.request
from pathlib import Path
import re, gc, random, time
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    """Nearest ancestor containing a data/ directory (works from notebooks/ or root)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def resolve(*names):
    """Local repo -> working dir -> GitHub. Returns (loaded_json, description)."""
    for n in names:
        if ROOT and (ROOT / "data" / n).exists():
            p = ROOT / "data" / n
            return json.loads(p.read_text(encoding="utf-8")), f"repo: {p}"
    for n in names:
        if Path(n).exists():
            return json.loads(Path(n).read_text(encoding="utf-8")), f"working dir: {n}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

# Outputs go to results/ when running inside the repo, else the working directory.
OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo -- will use working dir / GitHub)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "finetuned_path": None,   # C3 dropped: no end-to-end effect at n=68, so no split is needed
    "alpha": 0.8,
    "top_k": 5,
    "n_eval": 200,

    # Alternatives: "MBZUAI-Paris/Atlas-Chat-9B" (Darija-specialised),
    #               "Qwen/Qwen2.5-3B-Instruct" (faster, weaker)
    "llm": "Qwen/Qwen2.5-7B-Instruct",
    "load_4bit": True,          # requires CUDA + bitsandbytes; auto-disabled otherwise
    "max_new_tokens": 128,
    "judge_max_new_tokens": 40,
    "batch_size": 8,
    "max_prompt_tokens": 3072,

    "checkpoint": "gen_n200_checkpoint.json",
    "seed": 42,
}
CONFIG

### Load data

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c)
print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# No train/test split needed: condition C3 (fine-tuned encoder) is dropped,
# since it showed no end-to-end effect at n=68 (C3 - C2 = +0.000, CI [-0.088, +0.074]).
# Seeded shuffle kept so the evaluation order is reproducible.
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | QA {len(qa)} | evaluating {len(eval_qa)} (full benchmark, no holdout)")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

### Build contexts for all conditions, then free the encoder

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

def free_cuda():
    # Guarded: the original called torch.cuda.empty_cache() unconditionally.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def build(path):
    m = SentenceTransformer(path)
    e = m.encode([f"passage: {t}" for t in corpus_texts],
                 normalize_embeddings=True, batch_size=32, show_progress_bar=True)
    return m, np.asarray(e, "float32")

def retrieve_ids(model, emb, query, k):
    q = model.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = CONFIG["alpha"] * minmax(emb @ q) + (1 - CONFIG["alpha"]) * minmax(bm25_scores(query))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

contexts, K = {}, CONFIG["top_k"]

bm, be = build(CONFIG["base_encoder"])
contexts["C1_msa_base"]    = {q["id"]: retrieve_ids(bm, be, q["msa_query"], K) for q in eval_qa}
contexts["C2_darija_base"] = {q["id"]: retrieve_ids(bm, be, q["darija_query"], K) for q in eval_qa}
del bm, be
free_cuda()

if CONFIG["finetuned_path"] and os.path.exists(CONFIG["finetuned_path"]):
    fm, fe = build(CONFIG["finetuned_path"])
    contexts["C3_darija_finetuned"] = {q["id"]: retrieve_ids(fm, fe, q["darija_query"], K) for q in eval_qa}
    del fm, fe
    free_cuda()
else:
    print("Fine-tuned encoder not found - skipping C3.")

contexts["C4_oracle"] = {q["id"]: [q["source_chunk_id"]] for q in eval_qa}

# Encoders are released before the LLM loads, so both never sit in VRAM together.
for cond, d in contexts.items():
    hit = sum(1 for q in eval_qa if q["source_chunk_id"] in d[q["id"]]) / len(eval_qa)
    print(f"  {cond:<22} gold in context: {hit:.1%}")

json.dump(contexts, open(out("contexts.json"), "w", encoding="utf-8"), ensure_ascii=False)

### Load the LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

use_4bit = bool(CONFIG["load_4bit"]) and torch.cuda.is_available()
quant = None
if use_4bit:
    try:
        from transformers import BitsAndBytesConfig
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    except Exception as e:
        print(f"4-bit unavailable ({type(e).__name__}); loading unquantised.")
        use_4bit = False
elif CONFIG["load_4bit"]:
    print("load_4bit requested but CUDA is absent - bitsandbytes has no CPU/TPU backend, "
          "so the model loads unquantised.")

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"      # required for correct batched generation
tok.truncation_side = "left"   # overflow must drop CONTEXT, never the trailing instruction

# torch_dtype= was renamed dtype= in transformers 4.56, and the WRONG name is
# silently swallowed into **kwargs rather than raising. Dropping it entirely --
# as an earlier version of this notebook did -- leaves the non-quantised modules
# (embeddings, layer norms, lm_head) in float32 while the original ran them in
# float16. That changes generation numerics: greedy decoding then follows a
# different path and almost every answer comes out worded differently.
from transformers import __version__ as _tv
try:
    _v = tuple(int(x) for x in _tv.split(".")[:2])
except ValueError:
    _v = (0, 0)
LLM_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
kw = {("dtype" if _v >= (4, 56) else "torch_dtype"): LLM_DTYPE}
if quant:
    kw["quantization_config"] = quant
if torch.cuda.is_available():
    kw["device_map"] = "auto"
llm = AutoModelForCausalLM.from_pretrained(CONFIG["llm"], **kw)
llm.eval()
print(f"  transformers {_tv} -> passed '{list(kw)[0]}'={LLM_DTYPE}")
print(f"  parameter dtypes present: {sorted({str(p.dtype) for p in llm.parameters()})}")

print(f"Loaded {CONFIG['llm']}  (4-bit: {bool(quant)})")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - this will be slow")

TRUNCATED = 0   # counted, not silent

@torch.no_grad()
def chat_batch(prompts, max_new_tokens):
    """Batched chat completion. Batching is what makes local generation
    competitive with an API here."""
    global TRUNCATED
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True)
             for p in prompts]
    for t in texts:
        if len(tok(t)["input_ids"]) > CONFIG["max_prompt_tokens"]:
            TRUNCATED += 1
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
              max_length=CONFIG["max_prompt_tokens"]).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tok.decode(g, skip_special_tokens=True).strip() for g in gen]

print("Smoke test:", chat_batch(["\u0623\u062c\u0628 \u0628\u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629: \u0645\u0627 \u0639\u0627\u0635\u0645\u0629 \u0627\u0644\u0645\u063a\u0631\u0628\u061f"], 20)[0])

### Prompts and judge parsing

`parse_judge` is the fixed version. The original walked the reply line by line with
`if "مدعوم" … elif "مطابق" …`, so both verdicts on a single line lost the correctness one.
It now searches each field independently and reports whether the reply parsed at all, so a
malformed judge reply is visible instead of silently scoring (0, 0).

In [ ]:
GEN_PROMPT = """\u0623\u062c\u0628 \u0639\u0646 \u0627\u0644\u0633\u0624\u0627\u0644 \u0627\u0644\u062a\u0627\u0644\u064a \u0627\u0639\u062a\u0645\u0627\u062f\u0627 \u0641\u0642\u0637 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u0641\u0642\u0629.
\u0625\u0630\u0627 \u0644\u0645 \u062a\u0643\u0646 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0645\u0648\u062c\u0648\u062f\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635\u060c \u0642\u0644 \u0628\u0627\u0644\u0636\u0628\u0637: \u0627\u0644\u0645\u0639\u0644\u0648\u0645\u0629 \u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635
\u0644\u0627 \u062a\u0633\u062a\u0639\u0645\u0644 \u0623\u064a \u0645\u0639\u0631\u0641\u0629 \u062e\u0627\u0631\u062c\u064a\u0629. \u0623\u062c\u0628 \u0628\u062c\u0645\u0644\u0629 \u0648\u0627\u062d\u062f\u0629 \u0642\u0635\u064a\u0631\u0629 \u0641\u0642\u0637.

\u0627\u0644\u0646\u0635\u0648\u0635:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}

\u0627\u0644\u0625\u062c\u0627\u0628\u0629:"""

# Faithfulness and correctness are judged in ONE call to halve the work.
# They are separate constructs: an answer can faithfully report a passage that
# retrieval wrongly supplied, and so be faithful but incorrect.
JUDGE_PROMPT = """\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629: {gold}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629: {answer}

\u0623\u062c\u0628 \u0639\u0646 \u0633\u0624\u0627\u0644\u064a\u0646 \u0628\u062f\u0642\u0629:
1. \u0647\u0644 \u0643\u0644 \u0645\u0627 \u0648\u0631\u062f \u0641\u064a \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u062f\u0639\u0648\u0645 \u0635\u0631\u0627\u062d\u0629 \u0628\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629\u061f
2. \u0647\u0644 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u0637\u0627\u0628\u0642\u0629 \u0641\u064a \u0627\u0644\u0645\u0639\u0646\u0649 \u0644\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629\u061f \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0635\u064a\u0627\u063a\u0629 \u0645\u0642\u0628\u0648\u0644\u060c \u0623\u0645\u0627 \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0623\u0631\u0642\u0627\u0645 \u0623\u0648 \u0627\u0644\u0623\u0633\u0645\u0627\u0621 \u0623\u0648 \u0627\u0644\u062a\u0648\u0627\u0631\u064a\u062e \u0641\u063a\u064a\u0631 \u0645\u0642\u0628\u0648\u0644.

\u0623\u062c\u0628 \u0628\u0647\u0630\u0627 \u0627\u0644\u0634\u0643\u0644 \u0641\u0642\u0637 \u0648\u0628\u062f\u0648\u0646 \u0623\u064a \u0634\u0631\u062d:
\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627
\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627"""

REFUSAL = "\u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629"

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

_YES, _NO = "\u0646\u0639\u0645", "\u0644\u0627"
def _verdict(text, field):
    """Value of one labelled field, wherever it appears. None if absent or unusable.

    Searched per field rather than per line: the original's if/elif meant a reply
    with both verdicts on one line silently lost the second one. Whichever of
    yes/no appears first wins, so trailing commentary cannot flip the verdict,
    and a reply that merely echoes the template "\u0646\u0639\u0645/\u0644\u0627" counts as unparsed
    rather than as a spurious yes.
    """
    m = re.search(field + r"\s*[:\uFF1A]?\s*([^\n\u060c,]*)", text)
    if not m:
        return None
    v = m.group(1).strip()
    if re.fullmatch(_YES + r"\s*/\s*" + _NO, v):   # echoed the instruction verbatim
        return None
    iy, ino = v.find(_YES), v.find(_NO)
    if iy == -1 and ino == -1:
        return None
    if iy == -1:
        return 0
    if ino == -1:
        return 1
    return 1 if iy < ino else 0

def parse_judge(text):
    """-> (faithful, correct, parsed). `parsed` is 0 when the reply was unusable,
    which the original could not distinguish from a genuine double-negative."""
    t = (text or "").replace("\u060c", " ")
    f = _verdict(t, "\u0645\u062f\u0639\u0648\u0645")
    c = _verdict(t, "\u0645\u0637\u0627\u0628\u0642")
    parsed = int(f is not None and c is not None)
    return (f or 0), (c or 0), parsed

# Regression check for the bug that was fixed.
_c = [("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),   # normal two-line reply
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (0, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645 \u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),   # BOTH ON ONE LINE -- the original bug
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\u060c \u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),   # comma-separated
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627", (0, 0, 0)),  # echoed template -> unparsed
      ("", (0, 0, 0)),
      ("\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0635\u062d\u064a\u062d\u0629", (0, 0, 0))]
for txt, exp in _c:
    got = parse_judge(txt)
    assert got == exp, f"parse_judge regression: {txt!r} -> {got}, expected {exp}"
print("parse_judge: all regression cases pass (incl. both verdicts on one line)")

### Run (batched + checkpointed)

In [ ]:
from tqdm.auto import tqdm

CKPT = out(CONFIG["checkpoint"])
records = []
if os.path.exists(CKPT):
    records = json.load(open(CKPT, encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["condition"]) for r in records}

CONDITION_QUERY = {
    "C1_msa_base": "msa_query",
    "C2_darija_base": "darija_query",
    "C3_darija_finetuned": "darija_query",
    "C4_oracle": "darija_query",
}
byid = {q["id"]: q for q in eval_qa}
B = CONFIG["batch_size"]

for cond, ctx_map in contexts.items():
    qfield = CONDITION_QUERY[cond]
    todo = [q for q in eval_qa if (q["id"], cond) not in done]
    if not todo:
        continue
    print(f"\n=== {cond} ({len(todo)} to do) ===")

    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [ctx_map[q["id"]] for q in batch]

        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q[qfield])
                              for q, c in zip(batch, chunks)], CONFIG["max_new_tokens"])

        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c),
                                                   question=q["msa_query"],
                                                   gold=q["gold_answer"],
                                                   answer=a)
                               for q, c, a in zip(batch, chunks, answers)],
                              CONFIG["judge_max_new_tokens"])

        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok, parsed = parse_judge(v)
            records.append({
                "qid": q["id"], "condition": cond,
                "gold_in_context": int(q["source_chunk_id"] in c),
                "answer": a, "faithful": f, "correct": ok,
                "judge_parsed": parsed, "judge_raw": v,
                "refused": int(REFUSAL in (a or "")),
            })

        json.dump(records, open(CKPT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv(out("generation_local_raw.csv"), index=False)
print(f"\nComplete: {len(gen)} generations.")

# Both of these were silent in the original.
pf = 1 - gen.judge_parsed.mean()
print(f"Judge parse-failure rate: {pf:.1%}"
      + ("  <- metrics below are understated by this much" if pf > 0.01 else "  (negligible)"))
print(f"Prompts truncated: {TRUNCATED}"
      + ("  <- raise max prompt length" if TRUNCATED else "  (none)"))

### Results by condition

In [ ]:
print("=" * 88)
print("GENERATION RESULTS BY CONDITION")
print("=" * 88)
order = [c for c in CONDITION_QUERY if c in gen.condition.unique()]
summary = gen.groupby("condition").agg(
    n=("qid", "count"),
    gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"),
    correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"),
    judge_parsed=("judge_parsed", "mean"),
).reindex(order)

# Faithfulness among ANSWERED items only. A refusal is not "supported by the
# reference texts", so the judge scores refusals near-randomly -- in the committed
# n=68 run, 19 of 35 refusals were called faithful and 16 unfaithful. Reporting
# both columns keeps the headline number from being driven by refusal rate.
ans = gen[gen.refused == 0]
summary["faithfulness_answered"] = ans.groupby("condition")["faithful"].mean().reindex(order)
summary["correctness_answered"] = ans.groupby("condition")["correct"].mean().reindex(order)

print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv(out("generation_local_summary.csv"))

print("""
  gold_in_context        how often retrieval put the right passage before the generator
  faithfulness           answer supported by the context it was actually given
  correctness            answer matches the gold answer   <- what users care about
  refusal_rate           model declared the information absent
  judge_parsed           share of judge replies that parsed (1.000 = all good)
  *_answered             same metric excluding refusals -- see the note above
""")

### Paired comparisons with bootstrap CIs

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_cond, b_cond, col):
    a = gen[gen.condition == a_cond].set_index("qid")[col]
    b = gen[gen.condition == b_cond].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

print("=" * 88)
print("PAIRED COMPARISONS (95% CI)")
print("=" * 88)
rows = []
for a, b, label in [
    ("C1_msa_base", "C2_darija_base", "Does the dialect gap reach the answers?"),
    ("C3_darija_finetuned", "C2_darija_base", "Does the fine-tuned retriever help answers?"),
    ("C4_oracle", "C2_darija_base", "How much is lost to retrieval vs generation?"),
]:
    if a not in gen.condition.unique() or b not in gen.condition.unique():
        continue
    print(f"\n{label}   [{a} - {b}]")
    for col in ["correct", "faithful"]:
        d, lo, hi = paired(a, b, col)
        sig = "yes" if (lo > 0 or hi < 0) else "no"
        print(f"  {col:<10} {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  significant: {sig}")
        rows.append({"comparison": f"{a} - {b}", "metric": col,
                     "diff": d, "lo": lo, "hi": hi, "significant": sig})
pd.DataFrame(rows).to_csv(out("generation_local_comparisons.csv"), index=False)

### When retrieval misses, does the generator refuse or hallucinate?

The original printed a fixed paragraph asserting "low refusal + low correctness … the generator
does not notice the context is wrong and answers confidently anyway" — regardless of the data. In
the run it was printed under, the refusal rate on missing-gold rows was **0.737**, which is the
opposite behaviour. The conclusion is now read off the measured value.

In [ ]:
print("\n" + "=" * 88)
print("WHEN RETRIEVAL FAILS, WHAT DOES THE GENERATOR DO?")
print("=" * 88)
d2 = gen[gen.condition == "C2_darija_base"]
miss, hit = d2[d2.gold_in_context == 0], d2[d2.gold_in_context == 1]

if len(miss):
    for label, sub in [("MISSING", miss), ("PRESENT", hit)]:
        print(f"\nGold passage {label} ({len(sub)} cases):")
        print(f"  correctness   {sub.correct.mean():.3f}")
        print(f"  faithfulness  {sub.faithful.mean():.3f}")
        print(f"  refusal rate  {sub.refused.mean():.3f}")

    r = miss.refused.mean()
    print("\nInterpretation (derived from the numbers above, not assumed):")
    if r >= 0.5:
        print(f"  Refusal rate on missing-gold rows is {r:.3f} -- the generator mostly DETECTS")
        print("  that the retrieved context does not answer the question and declines, rather")
        print("  than inventing an answer. That is safe degradation: the end-to-end cost of a")
        print("  retrieval miss shows up as a non-answer, not as a confident falsehood.")
    elif r >= 0.2:
        print(f"  Refusal rate on missing-gold rows is {r:.3f} -- mixed. The generator catches")
        print("  some retrieval failures and answers through others.")
    else:
        print(f"  Refusal rate on missing-gold rows is {r:.3f} -- the failure the proposal")
        print("  predicted: the generator does not notice the context is wrong and answers")
        print("  confidently anyway.")
    print(f"\n  Correctness drops {hit.correct.mean():.3f} -> {miss.correct.mean():.3f} "
          f"when the gold passage is absent.")
else:
    print("Retrieval never missed in this sample - raise n_eval for this analysis.")

### Example failures for the paper

In [ ]:
print("\n=== Sample failures (dialect condition, incorrect answer) ===\n")
for _, r in gen[(gen.condition == "C2_darija_base") & (gen.correct == 0)].head(5).iterrows():
    q = byid[r["qid"]]
    print(f"Q (darija): {q['darija_query']}")
    print(f"Gold      : {q['gold_answer']}")
    print(f"Generated : {str(r['answer'])[:200]}")
    print(f"gold in context: {bool(r['gold_in_context'])} | faithful: {bool(r['faithful'])}"
          f" | refused: {bool(r['refused'])} | judge parsed: {bool(r['judge_parsed'])}")
    print("-" * 80)

print(f"\nAll outputs written to: {OUT_DIR.resolve()}")
for f in ["generation_local_raw.csv", "generation_local_summary.csv",
          "generation_local_comparisons.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab. Guarded so the notebook finishes cleanly anywhere.
try:
    from google.colab import files
    for f in ["generation_local_raw.csv", "generation_local_summary.csv",
              "generation_local_comparisons.csv"]:
        files.download(out(f))
except ImportError:
    print("\n(Not in Colab - files are on disk at the path above, no download needed.)")